<a href="https://colab.research.google.com/github/NBekhruzbek/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.7 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from datasets import load_dataset

In [ ]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens map', tokenizer.special_tokens_map)


# ---------------------------------------------------------------
# QUANTIZATION (kvantlash)
# Model og'irliklarini (weights) 16-bit o'rniga 4-bitda saqlaymiz.
# Natija: GPU xotirasi ~3-4 barobar kamayadi (~2.2 GB -> ~0.7 GB),
# aniqlik esa juda oz pasayadi.
# QLoRA fine-tuning uchun asos bo'ladi.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # og'irliklarni 4-bitda yuklash
    bnb_4bit_use_double_quant=True,         # scale konstantalarini ham kvantlash (qo'shimcha xotira tejash)
    bnb_4bit_compute_dtype=torch.bfloat16,  # hisoblashlar bfloat16 da bajariladi (saqlash 4-bit, hisob 16-bit)
    bnb_4bit_quant_type='nf4'               # NF4 (NormalFloat4): normal taqsimotdagi og'irliklar uchun optimal format
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is."
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)

first_block = model.model.layers[0]
print('frist_block:', first_block)
print('================================')
print(first_block.self_attn)
print('================================')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is. How does it work and what are its main benefits?
frist_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)


In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* insturction tuning

In [8]:
dataset = load_dataset("yahma/alpaca-cleaned", split='train')
dataset
dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [13]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        '### Input:\n'
        f"{input_text}\n\n"
        '### Response:\n'
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        '### Response:\n'
        f"{output_text}\n\n"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return{"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [15]:
dataset[0]["text"]

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n\n'

In [16]:
dataset = dataset.select(range(7000))

In [17]:
dataset = dataset.shuffle(seed=42) # seed =>  har safar bir xil malumotlarni shuffle qilib oladi

In [ ]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] EXACT SAME ORDER

In [21]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})